# Startup Valuation & Success Predictor
## Feature Engineering, Baseline Models & Advanced ML

This notebook walks through:
1. **Feature Engineering** - normalize, encode, create new features
2. **Baseline Models** - Linear & Logistic Regression
3. **Advanced Models** - XGBoost for regression & classification
4. **Evaluation** - metrics, cross-validation, SHAP explanations
5. **Save Models** - persist to disk for inference

## 1. Load & Explore Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix
import xgboost as xgb
import shap
import joblib
import os

# Load the dataset
df = pd.read_csv('../data/startup_funding.csv')
print(f"Dataset shape: {df.shape}")
print("\nFirst few rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

## 2. Feature Engineering

**What we'll do:**
- Separate numeric and categorical columns
- Scale numeric features (StandardScaler) → range 0–1
- One-hot encode categorical features (industry, country)
- Create derived features (e.g., funding_per_team_member)
- Drop non-predictive columns (startup_name)

In [ ]:
# Create a copy for processing
df_processed = df.copy()

# Separate numeric and categorical columns
numeric_cols = df_processed.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric columns: {numeric_cols}")
print(f"Categorical columns: {categorical_cols}")

# Drop startup_name (not useful for prediction)
if 'startup_name' in categorical_cols:
    df_processed = df_processed.drop('startup_name', axis=1)
    categorical_cols.remove('startup_name')

print(f"\nAfter dropping startup_name:")
print(f"Categorical: {categorical_cols}")

### 2.1 Create Derived Features

In [ ]:
# Create new meaningful features
df_processed['funding_per_team_member'] = df_processed['total_funding_usd'] / (df_processed['team_size'] + 1)  # +1 to avoid division by zero
df_processed['rounds_per_year'] = df_processed['funding_rounds'] / ((df_processed['months_since_founding'] / 12) + 1)
df_processed['years_since_founding'] = df_processed['months_since_founding'] / 12

print("New features created:")
print(df_processed[['funding_per_team_member', 'rounds_per_year', 'years_since_founding']].head(10))

### 2.2 Scale Numeric Features

In [ ]:
# Update numeric columns list
numeric_cols = df_processed.select_dtypes(include=[np.number]).columns.tolist()
# Remove target columns from features
target_cols = ['valuation_usd', 'is_successful']
feature_numeric_cols = [col for col in numeric_cols if col not in target_cols]

print(f"Numeric feature columns to scale: {feature_numeric_cols}")

# Initialize scaler
scaler = StandardScaler()
df_processed[feature_numeric_cols] = scaler.fit_transform(df_processed[feature_numeric_cols])

print("\nScaled numeric features (first 5 rows):")
print(df_processed[feature_numeric_cols].head())
print("\nMean ≈ 0, Std ≈ 1 (standardized):")
print(f"Mean: {df_processed[feature_numeric_cols].mean()}")
print(f"Std: {df_processed[feature_numeric_cols].std()}")

### 2.3 One-Hot Encode Categorical Features

In [ ]:
# One-hot encode categorical columns
df_encoded = pd.get_dummies(df_processed, columns=categorical_cols, drop_first=True)

print(f"Shape before encoding: {df_processed.shape}")
print(f"Shape after encoding: {df_encoded.shape}")
print(f"\nNew columns: {df_encoded.columns.tolist()}")
print(f"\nFirst 5 rows (selected columns):")
print(df_encoded.head())

### 2.4 Separate Features & Targets

In [ ]:
# Separate features and targets
X = df_encoded.drop(['valuation_usd', 'is_successful'], axis=1)
y_regression = df_encoded['valuation_usd']  # For valuation prediction
y_classification = df_encoded['is_successful']  # For success prediction

print(f"Features (X) shape: {X.shape}")
print(f"Target (valuation) shape: {y_regression.shape}")
print(f"Target (success) shape: {y_classification.shape}")
print(f"\nClass distribution (is_successful):")
print(y_classification.value_counts())

## 3. Train-Test Split

In [ ]:
# Split data: 80% train, 20% test
X_train, X_test, y_reg_train, y_reg_test = train_test_split(
    X, y_regression, test_size=0.2, random_state=42
)

X_train_clf, X_test_clf, y_clf_train, y_clf_test = train_test_split(
    X, y_classification, test_size=0.2, random_state=42
)

print(f"Regression split:")
print(f"  Train: {X_train.shape}, Test: {X_test.shape}")
print(f"\nClassification split:")
print(f"  Train: {X_train_clf.shape}, Test: {X_test_clf.shape}")

## 4. Baseline Models

**Linear Regression** - predict continuous valuation  
**Logistic Regression** - predict binary success/failure

In [ ]:
# ============ LINEAR REGRESSION (Valuation Prediction) ============
print("=" * 60)
print("LINEAR REGRESSION - Valuation Prediction")
print("=" * 60)

lr = LinearRegression()
lr.fit(X_train, y_reg_train)

# Predictions
y_pred_train = lr.predict(X_train)
y_pred_test = lr.predict(X_test)

# Metrics
r2_train = r2_score(y_reg_train, y_pred_train)
r2_test = r2_score(y_reg_test, y_pred_test)
rmse_train = np.sqrt(mean_squared_error(y_reg_train, y_pred_train))
rmse_test = np.sqrt(mean_squared_error(y_reg_test, y_pred_test))

print(f"\nTrain R² Score: {r2_train:.4f}")
print(f"Test R² Score:  {r2_test:.4f}")
print(f"Train RMSE:     ${rmse_train:,.0f}")
print(f"Test RMSE:      ${rmse_test:,.0f}")

# Feature importance (coefficients)
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': lr.coef_})
coef_df = coef_df.sort_values('Coefficient', ascending=False, key=abs)
print(f"\nTop 10 Important Features:")
print(coef_df.head(10))

In [ ]:
# ============ LOGISTIC REGRESSION (Success Prediction) ============
print("\n" + "=" * 60)
print("LOGISTIC REGRESSION - Success Prediction (Binary Classification)")
print("=" * 60)

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_clf, y_clf_train)

# Predictions
y_pred_clf_train = log_reg.predict(X_train_clf)
y_pred_clf_test = log_reg.predict(X_test_clf)

# Metrics
acc_train = accuracy_score(y_clf_train, y_pred_clf_train)
acc_test = accuracy_score(y_clf_test, y_pred_clf_test)

print(f"\nTrain Accuracy: {acc_train:.4f}")
print(f"Test Accuracy:  {acc_test:.4f}")
print(f"\nClassification Report (Test Set):")
print(classification_report(y_clf_test, y_pred_clf_test, target_names=['Not Successful', 'Successful']))

# Confusion Matrix
cm = confusion_matrix(y_clf_test, y_pred_clf_test)
print(f"\nConfusion Matrix:")
print(cm)

## 5. Advanced Models - XGBoost

**Why XGBoost?**
- Handles non-linear relationships better
- Usually outperforms linear models on tabular data
- Built-in feature importance
- Handles interactions automatically

In [ ]:
# ============ XGBOOST REGRESSOR (Valuation) ============
print("=" * 60)
print("XGBOOST REGRESSOR - Valuation Prediction")
print("=" * 60)

xgb_reg = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    verbosity=0
)

xgb_reg.fit(X_train, y_reg_train)

# Predictions
y_pred_xgb_train = xgb_reg.predict(X_train)
y_pred_xgb_test = xgb_reg.predict(X_test)

# Metrics
r2_xgb_train = r2_score(y_reg_train, y_pred_xgb_train)
r2_xgb_test = r2_score(y_reg_test, y_pred_xgb_test)
rmse_xgb_train = np.sqrt(mean_squared_error(y_reg_train, y_pred_xgb_train))
rmse_xgb_test = np.sqrt(mean_squared_error(y_reg_test, y_pred_xgb_test))

print(f"\nTrain R² Score: {r2_xgb_train:.4f}")
print(f"Test R² Score:  {r2_xgb_test:.4f}")
print(f"Train RMSE:     ${rmse_xgb_train:,.0f}")
print(f"Test RMSE:      ${rmse_xgb_test:,.0f}")

# Feature importance
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_reg.feature_importances_
}).sort_values('Importance', ascending=False)

print(f"\nTop 10 Important Features:")
print(importance_df.head(10))

In [ ]:
# ============ XGBOOST CLASSIFIER (Success) ============
print("\n" + "=" * 60)
print("XGBOOST CLASSIFIER - Success Prediction")
print("=" * 60)

xgb_clf = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    verbosity=0
)

xgb_clf.fit(X_train_clf, y_clf_train)

# Predictions
y_pred_xgb_clf_train = xgb_clf.predict(X_train_clf)
y_pred_xgb_clf_test = xgb_clf.predict(X_test_clf)

# Metrics
acc_xgb_train = accuracy_score(y_clf_train, y_pred_xgb_clf_train)
acc_xgb_test = accuracy_score(y_clf_test, y_pred_xgb_clf_test)

print(f"\nTrain Accuracy: {acc_xgb_train:.4f}")
print(f"Test Accuracy:  {acc_xgb_test:.4f}")
print(f"\nClassification Report (Test Set):")
print(classification_report(y_clf_test, y_pred_xgb_clf_test, target_names=['Not Successful', 'Successful']))

## 6. Model Comparison & Visualization

In [ ]:
# Compare regression models
print("REGRESSION MODEL COMPARISON (Test Set):")
print("-" * 50)
comparison_reg = pd.DataFrame({
    'Model': ['Linear Regression', 'XGBoost Regressor'],
    'R² Score': [r2_test, r2_xgb_test],
    'RMSE': [rmse_test, rmse_xgb_test]
})
print(comparison_reg)
print(f"\nBest Model: {comparison_reg.loc[comparison_reg['R² Score'].idxmax(), 'Model']}")

print("\n" + "=" * 50)
print("CLASSIFICATION MODEL COMPARISON (Test Set):")
print("-" * 50)
comparison_clf = pd.DataFrame({
    'Model': ['Logistic Regression', 'XGBoost Classifier'],
    'Accuracy': [acc_test, acc_xgb_test]
})
print(comparison_clf)
print(f"\nBest Model: {comparison_clf.loc[comparison_clf['Accuracy'].idxmax(), 'Model']}")

## 7. Cross-Validation

**Why?** Ensures model generalizes well (not just lucky on test set)

In [ ]:
# 5-Fold Cross-Validation
print("5-FOLD CROSS-VALIDATION:")
print("=" * 50)

# XGBoost Regressor CV
cv_scores_xgb_reg = cross_val_score(xgb_reg, X, y_regression, cv=5, scoring='r2')
print(f"\nXGBoost Regressor CV R² Scores: {cv_scores_xgb_reg}")
print(f"Mean R²: {cv_scores_xgb_reg.mean():.4f} (+/- {cv_scores_xgb_reg.std():.4f})")

# XGBoost Classifier CV
cv_scores_xgb_clf = cross_val_score(xgb_clf, X, y_classification, cv=5, scoring='accuracy')
print(f"\nXGBoost Classifier CV Accuracy Scores: {cv_scores_xgb_clf}")
print(f"Mean Accuracy: {cv_scores_xgb_clf.mean():.4f} (+/- {cv_scores_xgb_clf.std():.4f})")

## 8. SHAP - Model Explainability

**What is SHAP?**
- Shows which features influence predictions
- Helps understand model decisions
- "Why did the model predict this valuation?"

In [ ]:
# SHAP for XGBoost Regressor
print("Calculating SHAP values for regression model...")
explainer_reg = shap.TreeExplainer(xgb_reg)
shap_values_reg = explainer_reg.shap_values(X_test)

# SHAP for XGBoost Classifier
print("Calculating SHAP values for classification model...")
explainer_clf = shap.TreeExplainer(xgb_clf)
shap_values_clf = explainer_clf.shap_values(X_test)

print("Done!")

In [ ]:
# Feature importance via SHAP (Regression)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_reg, X_test, plot_type="bar", show=False)
plt.title('SHAP Feature Importance - Valuation Prediction')
plt.tight_layout()
plt.savefig('../outputs/shap_importance_regression.png')
plt.show()
print("Saved: outputs/shap_importance_regression.png")

In [ ]:
# Feature importance via SHAP (Classification)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_clf, X_test, plot_type="bar", show=False)
plt.title('SHAP Feature Importance - Success Prediction')
plt.tight_layout()
plt.savefig('../outputs/shap_importance_classification.png')
plt.show()
print("Saved: outputs/shap_importance_classification.png")

## 9. Save Models & Preprocessing Objects

In [ ]:
# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save models
joblib.dump(xgb_reg, '../models/xgboost_regressor.joblib')
joblib.dump(xgb_clf, '../models/xgboost_classifier.joblib')
joblib.dump(scaler, '../models/scaler.joblib')

print("Models saved:")
print("  ✓ models/xgboost_regressor.joblib")
print("  ✓ models/xgboost_classifier.joblib")
print("  ✓ models/scaler.joblib")

## 10. Summary & Next Steps

### What We Built:
- ✅ Feature engineering (scaling, encoding, derived features)
- ✅ Baseline models (Linear & Logistic Regression)
- ✅ Advanced models (XGBoost Regressor & Classifier)
- ✅ Model evaluation (R², Accuracy, Cross-validation)
- ✅ SHAP explanations (feature importance)
- ✅ Model persistence (saved to disk)

### Next Steps:
1. **Update `src/train.py`** to use these approaches for production
2. **Update `app.py`** (Streamlit) to use real trained models
3. **Test predictions** on new data
4. **Deploy** the final application

In [ ]:
# Example: Make a prediction on new data
# Create a sample startup
sample_startup = pd.DataFrame({
    'founded_year': [2020],
    'team_size': [15],
    'funding_rounds': [3],
    'total_funding_usd': [5000000],
    'has_vc_backing': [1],
    'months_since_founding': [48],
    'industry': ['AI/ML'],
    'country': ['USA']
})

print("Sample startup:")
print(sample_startup)

# Create derived features
sample_startup['funding_per_team_member'] = sample_startup['total_funding_usd'] / (sample_startup['team_size'] + 1)
sample_startup['rounds_per_year'] = sample_startup['funding_rounds'] / ((sample_startup['months_since_founding'] / 12) + 1)
sample_startup['years_since_founding'] = sample_startup['months_since_founding'] / 12

# Encode and scale
sample_encoded = pd.get_dummies(sample_startup, columns=['industry', 'country'], drop_first=True)

# Ensure same columns as training data
for col in X.columns:
    if col not in sample_encoded.columns:
        sample_encoded[col] = 0
sample_encoded = sample_encoded[X.columns]

# Scale
sample_scaled = sample_encoded.copy()
sample_scaled[feature_numeric_cols] = scaler.transform(sample_scaled[feature_numeric_cols])

# Predict
predicted_valuation = xgb_reg.predict(sample_scaled)[0]
success_prob = xgb_clf.predict_proba(sample_scaled)[0]

print(f"\n📊 PREDICTIONS:")
print(f"Predicted Valuation: ${predicted_valuation:,.0f}")
print(f"Success Probability: {success_prob[1]:.2%} (Not Successful: {success_prob[0]:.2%})")